*Module 6 of 9*

> **¿Prefieres español?** Abre [`06_variables_y_etiquetas.ipynb`](../es/06_variables_y_etiquetas.ipynb) — es el mismo módulo, en español.


# 📊 Module 6 — Parcels become a table + the ground truth

🧭 **Objectives** — turn each parcel from Module 5 into a **row of numbers**
(features) using **zonal statistics**, then attach **ground-truth** labels
from real field points with a **purity filter**. At the end you have exactly
what a classifier eats: a table of parcels × features, some of them labelled.

📚 **Features = a parcel as numbers.** A model cannot look at a picture; it
needs numbers. For each parcel we summarize the 13 layers under it into
**zonal statistics** — here the **mean** and **standard deviation** of every
band. So each parcel becomes a row of 26 numbers. (The production pipeline
computes far more — min, max, sums, across many months — hundreds of columns;
Module 9.)

📚 **Ground truth = the answer key.** To train a model we need parcels whose
crop we *actually know*. Those come from **field points**: 1,645 GPS
locations in the Yaqui Valley where someone recorded the real crop. We drop
each point onto its parcel. A **purity filter** keeps only parcels where all
the points inside agree — mixed parcels are ambiguous and would teach the
model wrong lessons.

![features](../../anim/en/06_features.svg)

![labels and purity](../../anim/en/07_labels_purity.svg)


## Rebuild the segmentation

Each course module runs on a fresh kernel, so we re-load the tile and
re-segment it (same parameters as Module 5) before extracting features.


In [ ]:
# Get the workshop tile (a few MB; cached after the first download)
import os, sys

async def get_file(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await get_file("crop_tile_384.tif")
print("Tile ready:", TILE)

In [ ]:
import numpy as np, rasterio
import scipy.ndimage, sklearn.cluster
import shepherd_wasm

with rasterio.open(TILE) as src:
    img = src.read()
    band_names = list(src.descriptions)

result = shepherd_wasm.doShepherdSegmentation(
    img, numClusters=30, minSegmentSize=50, imgNullVal=0, fixedKMeansInit=True)
seg = result.segimg.astype(np.int32)
n_seg = int(seg.max())
print(f"{n_seg} parcels to describe")

## Zonal statistics with np.bincount

For every band we want the mean and standard deviation of the pixels inside
each parcel. `np.bincount` does this fast: it sums values grouped by segment
id. From the sum and the sum-of-squares we get the mean and the standard
deviation for all parcels at once — vectorized, no Python loop over parcels.


In [ ]:
flat_seg = seg.ravel()
counts = np.bincount(flat_seg, minlength=n_seg + 1).astype(float)
counts[counts == 0] = 1     # avoid divide-by-zero for unused ids

features = np.zeros((n_seg + 1, len(band_names) * 2), dtype=np.float32)
for b in range(len(band_names)):
    vals = img[b].ravel().astype(np.float64)
    s1 = np.bincount(flat_seg, weights=vals,       minlength=n_seg + 1)
    s2 = np.bincount(flat_seg, weights=vals * vals, minlength=n_seg + 1)
    mean = s1 / counts
    var  = np.maximum(s2 / counts - mean**2, 0)
    features[:, 2*b], features[:, 2*b+1] = mean, np.sqrt(var)

feature_names = [f"{n}_{s}" for n in band_names for s in ("mean", "std")]
print(f"Feature table: {features.shape[0]-1} parcels x {features.shape[1]} features")
print("First few feature names:", feature_names[:4])

## Attach the ground truth (with the purity filter)

The label raster carries the real crop id at the field-point locations. For
each parcel that contains labelled pixels, we keep it for training **only if
every labelled pixel inside agrees** on the same class — that is the purity
filter. Then we count how many pure parcels we have per crop.


In [ ]:
import json

async def get_file(name):
    import os, sys
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url); open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request; urllib.request.urlretrieve(url, dest)
    return dest

LABELS = await get_file("crop_labels_384.tif")
NAMES  = await get_file("class_names.json")

with rasterio.open(LABELS) as src:
    lab = src.read(1)
class_names = {int(k): v for k, v in json.load(open(NAMES)).items()}

parcel_label = np.zeros(n_seg + 1, dtype=int)
for sid in np.unique(seg[lab > 0]):
    values = lab[(seg == sid) & (lab > 0)]
    uniq = np.unique(values)
    if len(uniq) == 1:                 # pure parcel -> usable for training
        parcel_label[sid] = uniq[0]

train_ids = np.flatnonzero(parcel_label)
print(f"Pure labelled parcels: {len(train_ids)} of {n_seg}")
for cid, cname in class_names.items():
    print(f"  {cname:12s}: {(parcel_label[train_ids] == cid).sum():3d} parcels")

## 🧪 Check yourself

**Why summarize each parcel to a mean and standard deviation instead of
feeding the model all its pixels?**

<details><summary>Show answer</summary>

The model classifies *parcels*, and it needs a fixed-length row of numbers
per parcel — but parcels have different pixel counts. Zonal statistics
(mean, std, ...) compress any parcel into the same set of features, and they
capture what matters: the parcel's typical value and how uniform it is.

</details>

**What does the purity filter throw away, and why is that a good thing for
training?**

<details><summary>Show answer</summary>

It discards parcels whose field points disagree on the crop (mixed or
mislabelled parcels). Training only on parcels with a single, agreed class
prevents teaching the model contradictory examples, which would blur every
class it learns.

</details>


## 🔭 Go deeper

Optional: these bilingual concept cards expand what you just learned
(prerequisite chains, lineage to fundamentals, curated references):

- [Zonal statistics](https://abxda.github.io/rs-learning-audio/?id=zonal-statistics)
- [Ground truth](https://abxda.github.io/rs-learning-audio/?id=ground-truth)
- [Training samples](https://abxda.github.io/rs-learning-audio/?id=training-samples)
- [Sample balancing](https://abxda.github.io/rs-learning-audio/?id=sample-balancing)



---

[← Previous · Module 5 — From pixels to parcels: segmentation](05_pixels_to_parcels.ipynb) · [Next → · Module 7 — Machine learning from zero](07_machine_learning_from_zero.ipynb)
